# Proyecto 1 - Hoppers con búsqueda adversarial

**CC3085 - Inteligencia Artificial**

## Parte 1: representación e interpretación del juego

Este notebook seguirá el mismo orden utilizado en el laboratorio de Tic-Tac-Toe y en
Lecture 0. Primero se define el juego mediante funciones independientes del agente:

| Función | Responsabilidad |
|---|---|
| `initial_state()` | Construir el tablero inicial de Hoppers. |
| `player(state)` | Indicar a qué jugador le corresponde mover. |
| `actions(state)` | Obtener todas las jugadas legales. |
| `result(state, action)` | Crear el estado posterior sin modificar el original. |
| `winner(state)` | Identificar al ganador, si existe. |
| `terminal(state)` | Indicar si la partida terminó. |
| `utility(state)` | Asignar `+1` a P1, `-1` a P2 y `0` si aún no hay ganador. |

En etapas posteriores se agregarán Minimax con poda alfa-beta, límite de profundidad,
heurística, control de tiempo e interfaz gráfica.


## 1. Reglas que representará el programa

- El tablero tiene 10 filas y 10 columnas.
- Cada jugador inicia con 15 piezas dentro de un campamento triangular.
- P1 comienza en la esquina superior izquierda y busca llegar a la esquina inferior derecha.
- P2 comienza en la esquina inferior derecha y busca llegar a la esquina superior izquierda.
- Una pieza puede dar un **paso** a cualquier casilla vecina vacía, incluyendo diagonales.
- También puede **saltar** una pieza propia o rival hacia la casilla vacía inmediatamente posterior.
- En la misma jugada se pueden encadenar varios saltos. Las piezas saltadas no se capturan.
- Un paso no puede combinarse con saltos dentro de la misma jugada.
- Durante una cadena no se permite repetir una casilla de aterrizaje. Esto evita ciclos.
- Se aplica la regla anti-bloqueo: un campamento objetivo lleno cuenta como completado aunque
  contenga alguna pieza del jugador que inició allí. Se exige al menos una pieza del jugador
  que está llegando para que nadie gane en el estado inicial.


In [1]:
""""Depth indica los niveles adicionales del árbol que Minimax debe explorar. Por ejemplo, si depth=2, el agente explorará dos niveles adicionales del árbol de juego.
depth = 2 → se estudia una jugada
depth = 1 → se estudia la respuesta o posibles respuestas del contrincantee
depth = 0 → se detiene y usa heuristic() 

Cada llamada recursiva debe de reducir por 1 el depth (asíL: depth - 1) hasta llegar a depth=0, donde se debe de usar la función heuristic() para evaluar el estado actual del juego.


La lógica detrás de esto es que el agente explore las opciones del juego hasta cierta profundidad que no prolongue el tiempo de ejecución excesivamente.
Cuando depth=0, el agente no tiene más información sobre las jugadas así que llama a la función heuristic() para evaluar el estado actual del juego y tomar una decisión basada en esa evaluación. 
La heurística no decide la jugada, sino que asigna un valor al estado alcanzado y ese valor regresa al árbol para que minimax compare las alternativas.
"""

'"Depth indica los niveles adicionales del árbol que Minimax debe explorar. Por ejemplo, si depth=2, el agente explorará dos niveles adicionales del árbol de juego.\ndepth = 2 → se estudia una jugada\ndepth = 1 → se estudia la respuesta o posibles respuestas del contrincantee\ndepth = 0 → se detiene y usa heuristic() \n\nCada llamada recursiva debe de reducir por 1 el depth (asíL: depth - 1) hasta llegar a depth=0, donde se debe de usar la función heuristic() para evaluar el estado actual del juego.\n\n\nLa lógica detrás de esto es que el agente explore las opciones del juego hasta cierta profundidad que no prolongue el tiempo de ejecución excesivamente.\nCuando depth=0, el agente no tiene más información sobre las jugadas así que llama a la función heuristic() para evaluar el estado actual del juego y tomar una decisión basada en esa evaluación. \nLa heurística no decide la jugada, sino que asigna un valor al estado alcanzado y ese valor regresa al árbol para que minimax compare las a

In [2]:
from dataclasses import dataclass
from typing import Optional
import time

BOARD_SIZE = 10
P1 = "P1"  # En la interfaz será la BYD TI-7, me gusta ese vehículo.
P2 = "P2"  # En la interfaz será la Land Cruiser Prado
EMPTY = None

Position = tuple[int, int]
Action = tuple[Position, ...]
Board = tuple[tuple[Optional[str], ...], ...]

# Campamento superior izquierdo: 5 + 4 + 3 + 2 + 1 = 15 casillas.
"""
-----
----
---
--
-  
"""
P1_CAMP = frozenset(
    (row, col)
    for row in range(5)
    for col in range(5 - row)
)

# El campamento de P2 es el reflejo del campamento de P1.
P2_CAMP = frozenset(
    (BOARD_SIZE - 1 - row, BOARD_SIZE - 1 - col)
    for row, col in P1_CAMP
)

# Movimiento horizontal, vertical y diagonal.
DIRECTIONS = tuple(
    (dr, dc)
    for dr in (-1, 0, 1)
    for dc in (-1, 0, 1)
    if (dr, dc) != (0, 0)
)

## Es una fotografía del tablero en cualquier estado de la partida. Board contiene la posición de las piezas y 
# para conocer qué jugador está en la posición i, j se utiliza board[i][j]. Turn indica si es turno de P1 o de P2. El tablero es inmutable, 
# por lo que para realizar un movimiento se debe crear un nuevo tablero.
@dataclass(frozen=True)
class State:
    """Estado completo: tablero inmutable y jugador al que le toca mover."""

    board: Board
    turn: str


### ¿Por qué el turno forma parte del estado?

En Tic-Tac-Toe se podía contar cuántas X y O había. En Hoppers las piezas no se eliminan,
por lo que siempre hay 15 piezas de cada jugador. El tablero por sí solo no permite conocer
el turno. Guardarlo explícitamente también hace que cada nodo del árbol de búsqueda contenga
toda la información necesaria.

El tablero y las acciones usan tuplas para que sean inmutables y, más adelante, puedan
usarse como llaves de una caché durante Minimax.


In [3]:
def initial_state() -> State:
    """Devuelve el estado inicial de Hoppers; P1 siempre mueve primero."""
    board = [[EMPTY for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]

    for row, col in P1_CAMP:
        board[row][col] = P1

    for row, col in P2_CAMP:
        board[row][col] = P2

    
    return State(
        board=tuple(tuple(row) for row in board),
        turn=P1,
    )


def player(state: State) -> str:
    """Devuelve el jugador al que le corresponde mover."""
    return state.turn


def other_player(current_player: str) -> str:
    """Devuelve el contrincante de current_player."""
    return P2 if current_player == P1 else P1


In [4]:
# auxiliar de pruebas
def check(label, condition):
    print(("PASS  " if condition is True else "MULA ") + label)
    return bool(condition)


_initial = initial_state()
check("el tablero es de 10 x 10", len(_initial.board) == 10 and all(len(row) == 10 for row in _initial.board))
check("P1 inicia con 15 piezas", sum(cell == P1 for row in _initial.board for cell in row) == 15)
check("P2 inicia con 15 piezas", sum(cell == P2 for row in _initial.board for cell in row) == 15)
check("P1 mueve primero", player(_initial) == P1)
check("cada campamento tiene 15 casillas", len(P1_CAMP) == len(P2_CAMP) == 15)


PASS  el tablero es de 10 x 10
PASS  P1 inicia con 15 piezas
PASS  P2 inicia con 15 piezas
PASS  P1 mueve primero
PASS  cada campamento tiene 15 casillas


True

## 2. `actions(state)`: pasos y saltos encadenados

Una acción guarda la ruta completa recorrida por una pieza. Sus dos primeras coordenadas
siempre representan origen y primer destino. Si existen más coordenadas, son los siguientes
aterrizajes de una cadena de saltos.

Para encontrar todas las cadenas se usa una búsqueda en profundidad pequeña que parte de
cada pieza del jugador. Cada prefijo legal también es una acción válida, porque el jugador
puede decidir detenerse después de cualquier salto.


In [5]:
def inside_board(position: Position) -> bool:
    """Indica si una coordenada pertenece al tablero."""
    row, col = position
    return 0 <= row < BOARD_SIZE and 0 <= col < BOARD_SIZE


def _jump_actions_from(state: State, origin: Position) -> set[Action]:
    """Encuentra todas las rutas de uno o más saltos desde origin."""
    board = [list(row) for row in state.board]
    piece = board[origin[0]][origin[1]]
    jump_actions = set()

    def explore(current: Position, path: Action, visited: frozenset[Position]):
        current_row, current_col = current

        for dr, dc in DIRECTIONS:
            middle = (current_row + dr, current_col + dc)
            landing = (current_row + 2 * dr, current_col + 2 * dc)

            if not inside_board(middle) or not inside_board(landing):
                continue

            middle_piece = board[middle[0]][middle[1]]
            landing_piece = board[landing[0]][landing[1]]

            if middle_piece is EMPTY or landing_piece is not EMPTY or landing in visited:
                continue

            # Se simula el salto para que el siguiente salto vea el tablero correcto.
            board[current_row][current_col] = EMPTY
            board[landing[0]][landing[1]] = piece

            new_path = path + (landing,)
            jump_actions.add(new_path)
            explore(landing, new_path, visited | {landing})

            # Deshacer permite explorar otra rama sin contaminarla.
            board[landing[0]][landing[1]] = EMPTY
            board[current_row][current_col] = piece

    explore(origin, (origin,), frozenset({origin}))
    return jump_actions


def actions(state: State) -> set[Action]:
    """Devuelve todos los pasos y todas las rutas de saltos legales."""
    if terminal(state):
        return set()

    legal_actions = set()

    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if state.board[row][col] != player(state):
                continue

            origin = (row, col)

            # Pasos de una casilla.
            for dr, dc in DIRECTIONS: 
                destination = (row + dr, col + dc)
                if inside_board(destination) and state.board[destination[0]][destination[1]] is EMPTY:
                    legal_actions.add((origin, destination))

            # Saltos simples y encadenados.
            legal_actions.update(_jump_actions_from(state, origin))

    return legal_actions


> Nota de diseño: `actions` llama a `terminal`, (se define más abajo).
 Esto es válido en Python porque la llamada ocurre al ejecutar la función, después de haber corrido todas las 
 celdas de definiciones..


## 3. `result(state, action)`: aplicar sin mutar

Minimax examinará muchas jugadas hipotéticas desde un mismo estado. Si `result` modificara
el tablero recibido, una rama alteraría las demás. Por eso se crea una copia, se valida la
acción contra `actions(state)` y se devuelve un `State` nuevo con el turno alternado.


In [6]:
def result(state: State, action: Action) -> State:
    """Devuelve el estado posterior a una acción legal sin modificar state."""
    if action not in actions(state):
        raise ValueError(f"Jugada ilegal: {action}")

    origin = action[0]
    destination = action[-1]
    board = [list(row) for row in state.board]

    board[origin[0]][origin[1]] = EMPTY
    board[destination[0]][destination[1]] = player(state)

    return State(
        board=tuple(tuple(row) for row in board),
        turn=other_player(player(state)),
    )


## 4. `winner`, `terminal` y `utility`

P1 intenta completar `P2_CAMP` y P2 intenta completar `P1_CAMP`. Con la regla anti-bloqueo,
un campamento se considera completo si todas sus casillas están ocupadas y al menos una
pertenece al jugador que está llegando. Las piezas rivales que quedaron bloqueando también
cuentan como ocupación.

Si una posición construida artificialmente hace que ambos campamentos estén completos,
se considera ganador al jugador que realizó la última jugada, es decir, al contrario de
`state.turn`.


In [7]:
def _completed_target_camp(state: State, candidate: str) -> bool:
    """Comprueba la meta con la regla anti-bloqueo elegida para el proyecto."""
    target_camp = P2_CAMP if candidate == P1 else P1_CAMP
    occupants = [state.board[row][col] for row, col in target_camp]

    return all(piece is not EMPTY for piece in occupants) and candidate in occupants


def winner(state: State) -> Optional[str]:
    """Devuelve P1, P2 o None si todavía no hay ganador."""
    previous_player = other_player(player(state))

    # Revisar primero a quien acaba de mover respeta la idea de "primero en completar".
    if _completed_target_camp(state, previous_player):
        return previous_player

    if _completed_target_camp(state, player(state)):
        return player(state)

    return None


def terminal(state: State) -> bool:
    """La partida termina cuando alguno de los jugadores completa su meta."""
    return winner(state) is not None

## Indica el ganador.
def utility(state: State) -> int:
    """Asigna +1 si gana P1, -1 si gana P2 y 0 si no existe ganador."""
    game_winner = winner(state)

    if game_winner == P1:
        return 1
    if game_winner == P2:
        return -1
    return 0


## 5. Pruebas de la Parte 1

Una de las buenas prácticas que aprendí haciendo los laboratorios en la clase de IA. Estas pruebas verifican los casos centrales de la rúbrica, que son el tablero y que los turnos sean los correctos. También se verifican los movimientos en ocho direcciones, salto sobre cualquiera de los dos colores, saltos
encadenados, ausencia de mutación, cambio de turno, jugadas inválidas y estados de victoria.


In [8]:
def state_from_pieces(p1_positions=(), p2_positions=(), turn=P1):
    """Auxiliar exclusivo para construir escenarios pequeños de prueba."""
    board = [[EMPTY for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]
    for row, col in p1_positions:
        board[row][col] = P1
    for row, col in p2_positions:
        board[row][col] = P2
    return State(tuple(tuple(row) for row in board), turn)


# Pasos en las ocho direcciones desde el centro.
step_state = state_from_pieces(p1_positions={(5, 5)})
expected_steps = {
    ((5, 5), (5 + dr, 5 + dc))
    for dr, dc in DIRECTIONS
}
check("una pieza central puede dar 8 pasos", expected_steps <= actions(step_state))

# Se puede saltar una pieza propia y una rival.
own_jump_state = state_from_pieces(p1_positions={(2, 2), (3, 3)})
rival_jump_state = state_from_pieces(p1_positions={(2, 2)}, p2_positions={(3, 3)})
check("se puede saltar una pieza propia", ((2, 2), (4, 4)) in actions(own_jump_state))
check("se puede saltar una pieza rival", ((2, 2), (4, 4)) in actions(rival_jump_state))

# Cadena: (0,0) salta (1,1), aterriza en (2,2), salta (3,3) y llega a (4,4).
chain_state = state_from_pieces(
    p1_positions={(0, 0), (1, 1)},
    p2_positions={(3, 3)},
)
chain = ((0, 0), (2, 2), (4, 4))
check("el salto simple de una cadena también es legal", ((0, 0), (2, 2)) in actions(chain_state))
check("se generan saltos encadenados", chain in actions(chain_state))

# result crea otro estado y conserva intacto el anterior.
after_chain = result(chain_state, chain)
check("result no muta el estado original", chain_state.board[0][0] == P1 and chain_state.board[4][4] is EMPTY)
check("result mueve la pieza al destino final", after_chain.board[0][0] is EMPTY and after_chain.board[4][4] == P1)
check("las piezas saltadas no se capturan", after_chain.board[1][1] == P1 and after_chain.board[3][3] == P2)
check("result cambia el turno", player(after_chain) == P2)

try:
    result(chain_state, ((0, 0), (0, 3)))
    invalid_action_rejected = False
except ValueError:
    invalid_action_rejected = True
check("result rechaza jugadas ilegales", invalid_action_rejected)

# El estado inicial no puede activar accidentalmente la regla anti-bloqueo.
check("el estado inicial no tiene ganador", winner(initial_state()) is None)
check("el estado inicial no es terminal", terminal(initial_state()) is False)

# P1 llena el campamento derecho, excepto una casilla que P2 sigue bloqueando.
p2_target = sorted(P2_CAMP)
anti_block_state = state_from_pieces(
    p1_positions=set(p2_target[:-1]),
    p2_positions={p2_target[-1]},
    turn=P2,
)
check("la regla anti-bloqueo reconoce la victoria de P1", winner(anti_block_state) == P1)
check("un estado ganador es terminal", terminal(anti_block_state) is True)
check("la victoria de P1 tiene utilidad +1", utility(anti_block_state) == 1)

# Caso simétrico para P2.
p1_target = sorted(P1_CAMP)
p2_win_state = state_from_pieces(
    p1_positions={p1_target[-1]},
    p2_positions=set(p1_target[:-1]),
    turn=P1,
)
check("la regla anti-bloqueo reconoce la victoria de P2", winner(p2_win_state) == P2)
check("la victoria de P2 tiene utilidad -1", utility(p2_win_state) == -1)


PASS  una pieza central puede dar 8 pasos
PASS  se puede saltar una pieza propia
PASS  se puede saltar una pieza rival
PASS  el salto simple de una cadena también es legal
PASS  se generan saltos encadenados
PASS  result no muta el estado original
PASS  result mueve la pieza al destino final
PASS  las piezas saltadas no se capturan
PASS  result cambia el turno
PASS  result rechaza jugadas ilegales
PASS  el estado inicial no tiene ganador
PASS  el estado inicial no es terminal
PASS  la regla anti-bloqueo reconoce la victoria de P1
PASS  un estado ganador es terminal
PASS  la victoria de P1 tiene utilidad +1
PASS  la regla anti-bloqueo reconoce la victoria de P2
PASS  la victoria de P2 tiene utilidad -1


True

NÍTIDOOOO
# SEGUNDA ETAPA




## Función Minimax 

In [9]:
def _state_after_legal_action(state, action):
    """
    Aplica una acción que ya fue obtenida mediante actions(state).
    No necesita volver a comprobar si es legal.
    """
    origin = action[0]
    destination = action[-1]

    board = [list(row) for row in state.board]

    board[origin[0]][origin[1]] = EMPTY
    board[destination[0]][destination[1]] = player(state)

    return State(
        board=tuple(tuple(row) for row in board),
        turn=other_player(player(state))
    )

In [10]:
def _ordered_successors(state):
    """
    Devuelve pares (acción, nuevo_estado).
    Las victorias inmediatas se colocan primero.
    """
    successors = []

    for action in actions(state):
        child = _state_after_legal_action(state, action)

        immediate_win = winner(child) == player(state)
        priority = 0 if immediate_win else 1

        successors.append((priority, action, child))

    successors.sort(key=lambda item: (item[0], item[1]))

    return [
        (action, child)
        for _, action, child in successors
    ]

# Función heurística
#### Definición de pesos de la función


In [11]:
PROGRESS_WEIGHT = 0.60
CAMP_WEIGHT = 0.40



Progress weigth mide el avance general de varias piezas, mientras que camp mide el avance para entrar al campamento rival. Ninguna distingue entre avanzar varias piezas en general o solamente si tienen el mismo progreso total. 

In [12]:
## DISTANCIA NORMALIZADA ES LA MÁXIMA DISTANCIA = 18
def piece_progress(position, candidate):
    """ Calcula el progreso de una pieza hacia el campamento contrincante"""
    row, col = position
    maximum_distance = 2*(BOARD_SIZE - 1)  # Distancia máxima posible en el tablero

    if candidate == P1:
        return (row + col) / maximum_distance

    return(
        ( (BOARD_SIZE - 1 - row) + (BOARD_SIZE - 1 - col) ) / maximum_distance
    )

In [13]:
check(
    "P1 en la esquina inicial tiene progreso 0",
    piece_progress((0, 0), P1) == 0
)

check(
    "P1 en el centro tiene progreso 0.5",
    piece_progress((4, 5), P1) == 0.5
)

check(
    "P1 en la esquina objetivo tiene progreso 1",
    piece_progress((9, 9), P1) == 1
)
check(
    "P2 en la esquina inicial tiene progreso 0",
    piece_progress((9, 9), P2) == 0
)

check(
    "P2 en la esquina objetivo tiene progreso 1",
    piece_progress((0, 0), P2) == 1
)

PASS  P1 en la esquina inicial tiene progreso 0
PASS  P1 en el centro tiene progreso 0.5
PASS  P1 en la esquina objetivo tiene progreso 1
PASS  P2 en la esquina inicial tiene progreso 0
PASS  P2 en la esquina objetivo tiene progreso 1


True

## Función del progreso de todas las piezas


In [14]:
def total_progress(state, candidate):
    """Suma el progreso de todas las piezas de candidate."""
    total = 0.0

    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if state.board[row][col] == candidate:
                total += piece_progress((row, col), candidate)

    return total

### Función para comparar el progreso de los jugadores

In [15]:
def progress_balance(state):
    """Compara el progreso total de P1 contra el de P2."""
    p1_progress = total_progress(state, P1)
    p2_progress = total_progress(state, P2)

    return (p1_progress - p2_progress) / 15
## se normaliza para que los valores estén entre -1 y 1, ya que el progreso máximo de un jugador es 15 (todas sus piezas en el campamento contrario).

### función para contar piezas en target

In [16]:
def pieces_in_target_camp(state, candidate):
    """Cuenta las piezas propias dentro del campamento objetivo."""
    target_camp = P2_CAMP if candidate == P1 else P1_CAMP

    return sum(
        state.board[row][col] == candidate
        for row, col in target_camp
    )

### Balance de ocupación del campamento

In [17]:
def camp_balance(state):
    """Compara la ocupación de los campamentos objetivo."""
    p1_in_target = pieces_in_target_camp(state, P1)
    p2_in_target = pieces_in_target_camp(state, P2)

    return (p1_in_target - p2_in_target) / 15

## Función heurística


In [18]:
def heuristic(state):
    """Estima qué jugador tiene ventaja."""
    if terminal(state):
        return float(utility(state))

    return (
        PROGRESS_WEIGHT * progress_balance(state)
        + CAMP_WEIGHT * camp_balance(state)
    )

### Prueba de heurística

In [19]:


initial_score = heuristic(initial_state())

check(
    "el estado inicial tiene una evaluación equilibrada",
    abs(initial_score) < 1e-9
)

balanced_state = state_from_pieces(
    p1_positions={(0, 0)},
    p2_positions={(9, 9)}
)

p1_advanced_state = state_from_pieces(
    p1_positions={(1, 1)},
    p2_positions={(9, 9)}
)

p2_advanced_state = state_from_pieces(
    p1_positions={(0, 0)},
    p2_positions={(8, 8)}
)

check(
    "un estado equilibrado vale aproximadamente 0",
    abs(heuristic(balanced_state)) < 1e-9
)

check(
    "avanzar P1 produce una evaluación positiva",
    heuristic(p1_advanced_state) > 0
)

check(
    "avanzar P2 produce una evaluación negativa",
    heuristic(p2_advanced_state) < 0
)

print("Evaluación inicial:", initial_score)
print("P1 adelantado:", heuristic(p1_advanced_state))
print("P2 adelantado:", heuristic(p2_advanced_state))

PASS  el estado inicial tiene una evaluación equilibrada
PASS  un estado equilibrado vale aproximadamente 0
PASS  avanzar P1 produce una evaluación positiva
PASS  avanzar P2 produce una evaluación negativa
Evaluación inicial: 1.7763568394002505e-17
P1 adelantado: 0.004444444444444444
P2 adelantado: -0.004444444444444444


## Control de tiempo


In [20]:
# Controles de tiempo
class SearchTimeout(Exception):
    """Indica que Minimax agotó el tiempo disponible."""
    pass


def check_time(deadline):
    """Detiene la búsqueda cuando se alcanza el tiempo límite."""
    if time.perf_counter() >= deadline:
        raise SearchTimeout

In [21]:

@dataclass
class SearchStats:
    """Contadores para observar el trabajo realizado por Minimax."""
    visited_nodes: int = 0
    pruned_branches: int = 0

### Función MAX 
Donde p1 trata de obtener el valor más grande (como en el diagrma del árbol).

In [22]:
def _max_value_ab(
    state,
    depth,
    alpha,
    beta,
    stats,
    deadline
):
    """Busca el valor más alto que P1 puede garantizar."""
    check_time(deadline)
    stats.visited_nodes += 1

    if terminal(state):
        return float(utility(state))

    if depth == 0:
        return heuristic(state)

    value = float("-inf")

    for _, child in _ordered_successors(state):
        check_time(deadline)

        child_value = _min_value_ab(
            child,
            depth - 1,
            alpha,
            beta,
            stats,
            deadline
        )

        value = max(value, child_value)

        if value >= beta:
            stats.pruned_branches += 1
            return value

        alpha = max(alpha, value)

    return value

#### MIn value

In [23]:
def _min_value_ab(
    state,
    depth,
    alpha,
    beta,
    stats,
    deadline
):
    """Busca el valor más bajo que P2 puede garantizar."""
    check_time(deadline)
    stats.visited_nodes += 1

    if terminal(state):
        return float(utility(state))

    if depth == 0:
        return heuristic(state)

    value = float("inf")

    for _, child in _ordered_successors(state):
        check_time(deadline)

        child_value = _max_value_ab(
            child,
            depth - 1,
            alpha,
            beta,
            stats,
            deadline
        )

        value = min(value, child_value)

        if value <= alpha:
            stats.pruned_branches += 1
            return value

        beta = min(beta, value)

    return value

### Mejor jugada al podar

In [24]:
def minimax_alpha_beta(
    state,
    depth=2,
    return_stats=False,
    deadline=None
):
    """Ejecuta Minimax a una profundidad específica."""
    if depth < 1:
        raise ValueError("depth debe ser al menos 1")

    # Sin deadline, la búsqueda funciona como antes.
    if deadline is None:
        deadline = float("inf")

    check_time(deadline)

    if terminal(state):
        if return_stats:
            return None, SearchStats()
        return None

    stats = SearchStats()
    best_action = None

    if player(state) == P1:
        best_value = float("-inf")
        alpha = float("-inf")
        beta = float("inf")

        for action, child in _ordered_successors(state):
            check_time(deadline)

            value = _min_value_ab(
                child,
                depth - 1,
                alpha,
                beta,
                stats,
                deadline
            )

            if value > best_value:
                best_value = value
                best_action = action

            alpha = max(alpha, best_value)

            if best_value == 1:
                break

    else:
        best_value = float("inf")
        alpha = float("-inf")
        beta = float("inf")

        for action, child in _ordered_successors(state):
            check_time(deadline)

            value = _max_value_ab(
                child,
                depth - 1,
                alpha,
                beta,
                stats,
                deadline
            )

            if value < best_value:
                best_value = value
                best_action = action

            beta = min(beta, best_value)

            if best_value == -1:
                break

    if return_stats:
        return best_action, stats

    return best_action

### Profundidad iterativa

In [25]:
def iterative_deepening_agent(
    state,
    time_limit=30,
    max_depth=None,
    return_info=False
):
    """Busca con profundidades crecientes hasta agotar el tiempo."""
    if time_limit <= 0:
        raise ValueError("time_limit debe ser mayor que 0")

    if terminal(state):
        if return_info:
            return None, 0, SearchStats()
        return None

    legal_actions = sorted(actions(state))

    if not legal_actions:
        if return_info:
            return None, 0, SearchStats()
        return None

    deadline = time.perf_counter() + time_limit

    # Jugada de respaldo por si no termina depth=1.
    best_action = legal_actions[0]
    completed_depth = 0
    total_stats = SearchStats()

    depth = 1

    while max_depth is None or depth <= max_depth:
        try:
            action, stats = minimax_alpha_beta(
                state,
                depth=depth,
                return_stats=True,
                deadline=deadline
            )

            # Esta profundidad terminó completamente.
            best_action = action
            completed_depth = depth

            total_stats.visited_nodes += stats.visited_nodes
            total_stats.pruned_branches += stats.pruned_branches

            depth += 1

        except SearchTimeout:
            break

    if return_info:
        return best_action, completed_depth, total_stats

    return best_action

In [26]:
def minimax_agent(state, time_limit=30):
    """Agente principal con profundidad iterativa y límite de tiempo."""
    return iterative_deepening_agent(
        state,
        time_limit=time_limit
    )

#### Pruebas

In [27]:

initial = initial_state()

start_time = time.perf_counter()

action, completed_depth, stats = iterative_deepening_agent(
    initial,
    time_limit=1,
    return_info=True
)

elapsed_time = time.perf_counter() - start_time

print("Jugada elegida:", action)
print("Jugada legal:", action in actions(initial))
print("Profundidad completada:", completed_depth)
print("Nodos visitados:", stats.visited_nodes)
print("Ramas podadas:", stats.pruned_branches)
print("Tiempo utilizado:", elapsed_time)

check(
    "el agente con tiempo devuelve una jugada legal",
    action in actions(initial)
)

Jugada elegida: ((2, 2), (3, 3))
Jugada legal: True
Profundidad completada: 3
Nodos visitados: 3189
Ramas podadas: 205
Tiempo utilizado: 1.0013258999970276
PASS  el agente con tiempo devuelve una jugada legal


True

In [28]:
initial = initial_state()

action, stats = minimax_alpha_beta(initial, depth=1, return_stats=True)

print("Jugada ", action)
print("Jugada legal en: ", str(action in actions(initial)))

print("Visitó estos nodos: ", stats.visited_nodes)
print("Podó estas ramas: ", stats.pruned_branches)

check("Minimax da una jugada legal?", action in actions(initial))



Jugada  ((0, 1), (2, 3))
Jugada legal en:  True
Visitó estos nodos:  32
Podó estas ramas:  0
PASS  Minimax da una jugada legal?


True

In [29]:
action_depth_2, stats_depth_2 = minimax_alpha_beta(
    initial,
    depth=2,
    return_stats=True
)

print("Jugada con profundidad 2:", action_depth_2)
print("La jugada es legal:", action_depth_2 in actions(initial))
print("Nodos visitados:", stats_depth_2.visited_nodes)
print("Ramas podadas:", stats_depth_2.pruned_branches)

check(
    "Minimax con depth=2 devuelve una jugada legal",
    action_depth_2 in actions(initial)
)


Jugada con profundidad 2: ((0, 1), (2, 3))
La jugada es legal: True
Nodos visitados: 234
Ramas podadas: 31
PASS  Minimax con depth=2 devuelve una jugada legal


True

Lo que hace con depth 1 es generar y probar todas las posibles jugadas de p1 en 0,0 para llegar a depth = 0. Luego evalúa cada estado con heuristic (que da el coeficiente de progreso e invasión en forma de coeficiente). Con ello escoge el estado con valor más alto.

#### Uso

In [30]:
def one_move_from_winning(candidate):
    """
    Construye un estado donde candidate puede ganar
    realizando una sola jugada.
    """
    target = P2_CAMP if candidate == P1 else P1_CAMP

    for empty_position in sorted(target):
        empty_row, empty_col = empty_position

        for dr, dc in DIRECTIONS:
            origin = (
                empty_row - dr,
                empty_col - dc
            )

            if inside_board(origin) and origin not in target:
                own_positions = set(target) - {empty_position}
                own_positions.add(origin)

                if candidate == P1:
                    return state_from_pieces(
                        p1_positions=own_positions,
                        turn=P1
                    )

                return state_from_pieces(
                    p2_positions=own_positions,
                    turn=P2
                )

    raise ValueError(
        "No se pudo construir el escenario de prueba"
    )

In [31]:
state = one_move_from_winning(P1)

action, stats = minimax_alpha_beta(
    state,
    return_stats=True
)

print("Jugada elegida:", action)
print("Nodos visitados:", stats.visited_nodes)
print("Ramas podadas:", stats.pruned_branches)



Jugada elegida: ((5, 8), (5, 9))
Nodos visitados: 1
Ramas podadas: 0


# Parte 3
## Controlador de la partida

### Visualización textual del tablero

In [32]:
def display_board(state):
    """Muestra el tablero y el turno actual en la consola."""
    symbols = {
        EMPTY: ".",
        P1: "B",  #TI-7
        P2: "L"   # Prado
    }

    # Números de las columnas.
    print("   " + " ".join(str(col) for col in range(BOARD_SIZE)))

    # Contenido de cada fila.
    for row in range(BOARD_SIZE):
        cells = [
            symbols[state.board[row][col]]
            for col in range(BOARD_SIZE)
        ]

        print(f"{row:2} " + " ".join(cells))

    if terminal(state):
        print("Ganador:", winner(state))
    else:
        print("Turno:", player(state))

In [33]:
state = initial_state()
display_board(state)

   0 1 2 3 4 5 6 7 8 9
 0 B B B B B . . . . .
 1 B B B B . . . . . .
 2 B B B . . . . . . .
 3 B B . . . . . . . .
 4 B . . . . . . . . .
 5 . . . . . . . . . L
 6 . . . . . . . . L L
 7 . . . . . . . L L L
 8 . . . . . . L L L L
 9 . . . . . L L L L L
Turno: P1


In [34]:
action = sorted(actions(state))[2]
new_state = result(state, action)

print("Acción realizada:", action)
display_board(new_state)

Acción realizada: ((0, 3), (0, 5))
   0 1 2 3 4 5 6 7 8 9
 0 B B B . B B . . . .
 1 B B B B . . . . . .
 2 B B B . . . . . . .
 3 B B . . . . . . . .
 4 B . . . . . . . . .
 5 . . . . . . . . . L
 6 . . . . . . . . L L
 7 . . . . . . . L L L
 8 . . . . . . L L L L
 9 . . . . . L L L L L
Turno: P2


### Agente humano

In [35]:
def human_agent(state):
    """Solicita al usuario una jugada legal y la devuelve."""
    legal_actions = sorted(actions(state))

    if not legal_actions:
        return None

    print("\nJugadas legales:")

    for index, action in enumerate(legal_actions, start=1):
        print(f"{index}: {action}")

    while True:
        try:
            selection = int(
                input("Selecciona el número de la jugada: ")
            )

            if 1 <= selection <= len(legal_actions):
                return legal_actions[selection - 1]

            print(
                f"Escribe un número entre 1 y "
                f"{len(legal_actions)}."
            )

        except ValueError:
            print("Debes escribir un número entero.")

In [36]:
state = initial_state()
display_board(state)

human_action = human_agent(state)

print("Jugada seleccionada:", human_action)
print("¿Es legal?", human_action in actions(state))

   0 1 2 3 4 5 6 7 8 9
 0 B B B B B . . . . .
 1 B B B B . . . . . .
 2 B B B . . . . . . .
 3 B B . . . . . . . .
 4 B . . . . . . . . .
 5 . . . . . . . . . L
 6 . . . . . . . . L L
 7 . . . . . . . L L L
 8 . . . . . . L L L L
 9 . . . . . L L L L L
Turno: P1

Jugadas legales:
1: ((0, 1), (2, 3))
2: ((0, 2), (2, 4))
3: ((0, 3), (0, 5))
4: ((0, 3), (1, 4))
5: ((0, 3), (2, 3))
6: ((0, 4), (0, 5))
7: ((0, 4), (1, 4))
8: ((0, 4), (1, 5))
9: ((1, 0), (3, 2))
10: ((1, 1), (3, 3))
11: ((1, 2), (1, 4))
12: ((1, 2), (2, 3))
13: ((1, 2), (3, 2))
14: ((1, 3), (1, 4))
15: ((1, 3), (2, 3))
16: ((1, 3), (2, 4))
17: ((2, 0), (4, 2))
18: ((2, 1), (2, 3))
19: ((2, 1), (3, 2))
20: ((2, 1), (4, 1))
21: ((2, 2), (2, 3))
22: ((2, 2), (3, 2))
23: ((2, 2), (3, 3))
24: ((3, 0), (3, 2))
25: ((3, 0), (4, 1))
26: ((3, 0), (5, 0))
27: ((3, 1), (3, 2))
28: ((3, 1), (4, 1))
29: ((3, 1), (4, 2))
30: ((4, 0), (4, 1))
31: ((4, 0), (5, 0))
32: ((4, 0), (5, 1))
Jugada seleccionada: ((0, 1), (2, 3))
¿Es legal? True


### Controlador para humano vs. humano y humano vs. agente 

In [37]:
def play_game(
    p1_agent,
    p2_agent,
    start_state=None,
    show_board=True,
    max_turns=None
):
    """
    Ejecuta una partida usando un agente para P1 y otro para P2.

    Cada agente debe recibir un estado y devolver una acción.
    """
    state = initial_state() if start_state is None else start_state

    agents = {
        P1: p1_agent,
        P2: p2_agent
    }

    turn_count = 0

    while not terminal(state):
        if max_turns is not None and turn_count >= max_turns:
            print("Se alcanzó el límite de turnos de prueba.")
            return state

        if show_board:
            print(f"\nTurno número {turn_count + 1}")
            display_board(state)

        current_player = player(state)
        selected_agent = agents[current_player]

        action = selected_agent(state)

        if action is None:
            raise RuntimeError(
                f"{current_player} no devolvió ninguna jugada."
            )

        if action not in actions(state):
            raise ValueError(
                f"{current_player} devolvió una jugada ilegal: {action}"
            )

        print(f"{current_player} juega: {action}")

        state = result(state, action)
        turn_count += 1

    print("\nPartida terminada")
    display_board(state)
    print("Cantidad de turnos:", turn_count)

    return state

#### Pruebas

In [38]:
def first_legal_agent(state):
    """Agente básico utilizado solamente para probar el controlador."""
    return sorted(actions(state))[0]

test_state = play_game(
    p1_agent=first_legal_agent,
    p2_agent=first_legal_agent,
    show_board=True,
    max_turns=4
)

check(
    "el controlador completó cuatro turnos",
    player(test_state) == P1
)


Turno número 1
   0 1 2 3 4 5 6 7 8 9
 0 B B B B B . . . . .
 1 B B B B . . . . . .
 2 B B B . . . . . . .
 3 B B . . . . . . . .
 4 B . . . . . . . . .
 5 . . . . . . . . . L
 6 . . . . . . . . L L
 7 . . . . . . . L L L
 8 . . . . . . L L L L
 9 . . . . . L L L L L
Turno: P1
P1 juega: ((0, 1), (2, 3))

Turno número 2
   0 1 2 3 4 5 6 7 8 9
 0 B . B B B . . . . .
 1 B B B B . . . . . .
 2 B B B B . . . . . .
 3 B B . . . . . . . .
 4 B . . . . . . . . .
 5 . . . . . . . . . L
 6 . . . . . . . . L L
 7 . . . . . . . L L L
 8 . . . . . . L L L L
 9 . . . . . L L L L L
Turno: P2
P2 juega: ((5, 9), (4, 8))

Turno número 3
   0 1 2 3 4 5 6 7 8 9
 0 B . B B B . . . . .
 1 B B B B . . . . . .
 2 B B B B . . . . . .
 3 B B . . . . . . . .
 4 B . . . . . . . L .
 5 . . . . . . . . . .
 6 . . . . . . . . L L
 7 . . . . . . . L L L
 8 . . . . . . L L L L
 9 . . . . . L L L L L
Turno: P1
P1 juega: ((0, 0), (0, 1))

Turno número 4
   0 1 2 3 4 5 6 7 8 9
 0 . B B B B . . . . .
 1 B B B B . . . . .

True

Nota: el controlador tiene que verificar para cada jugada que la acción pertenezca a las acciones legales de action(state) debido a que pueden haber, tanto humanos como agentes de IA, que intenten hacer un movimiento que nada que ver.

### Modo humano contra agente

In [39]:
def play_human_vs_agent(
    human_player=P1,
    time_limit=30
):
    """Inicia una partida entre una persona y Minimax."""
    if human_player not in (P1, P2):
        raise ValueError("human_player debe ser P1 o P2")

    def ai_agent(state):
        return minimax_agent(
            state,
            time_limit=time_limit
        )

    if human_player == P1:
        p1_agent = human_agent
        p2_agent = ai_agent
    else:
        p1_agent = ai_agent
        p2_agent = human_agent

    return play_game(
        p1_agent=p1_agent,
        p2_agent=p2_agent,
        show_board=True
    )

In [40]:
""" final_state = play_human_vs_agent(
    human_player=P1,
    time_limit=1 )
    """

' final_state = play_human_vs_agent(\n    human_player=P1,\n    time_limit=1 )\n    '

### Agente vs. Agente

In [41]:
def play_agent_vs_agent(
    time_limit=30,
    max_turns=None,
    show_board=True
):
    """Inicia una partida de Minimax contra Minimax."""

    def ai_agent(state):
        return minimax_agent(
            state,
            time_limit=time_limit
        )

    return play_game(
        p1_agent=ai_agent,
        p2_agent=ai_agent,
        show_board=show_board,
        max_turns=max_turns
    )

In [42]:
test_state = play_agent_vs_agent(
    time_limit=1,
    max_turns=5,
    show_board=True
)


Turno número 1
   0 1 2 3 4 5 6 7 8 9
 0 B B B B B . . . . .
 1 B B B B . . . . . .
 2 B B B . . . . . . .
 3 B B . . . . . . . .
 4 B . . . . . . . . .
 5 . . . . . . . . . L
 6 . . . . . . . . L L
 7 . . . . . . . L L L
 8 . . . . . . L L L L
 9 . . . . . L L L L L
Turno: P1
P1 juega: ((2, 2), (3, 3))

Turno número 2
   0 1 2 3 4 5 6 7 8 9
 0 B B B B B . . . . .
 1 B B B B . . . . . .
 2 B B . . . . . . . .
 3 B B . B . . . . . .
 4 B . . . . . . . . .
 5 . . . . . . . . . L
 6 . . . . . . . . L L
 7 . . . . . . . L L L
 8 . . . . . . L L L L
 9 . . . . . L L L L L
Turno: P2
P2 juega: ((7, 7), (6, 6))

Turno número 3
   0 1 2 3 4 5 6 7 8 9
 0 B B B B B . . . . .
 1 B B B B . . . . . .
 2 B B . . . . . . . .
 3 B B . B . . . . . .
 4 B . . . . . . . . .
 5 . . . . . . . . . L
 6 . . . . . . L . L L
 7 . . . . . . . . L L
 8 . . . . . . L L L L
 9 . . . . . L L L L L
Turno: P1
P1 juega: ((0, 0), (2, 2), (4, 4))

Turno número 4
   0 1 2 3 4 5 6 7 8 9
 0 . B B B B . . . . .
 1 B B B B .

### Determinar ganador de partida

In [43]:
def depth_one_agent(state):
    """Minimax de profundidad 1 para pruebas rápidas."""
    return minimax_alpha_beta(
        state,
        depth=1
    )

In [44]:
near_win_state = one_move_from_winning(P1)

final_state = play_game(
    p1_agent=depth_one_agent,
    p2_agent=first_legal_agent,
    start_state=near_win_state,
    show_board=True,
    max_turns=2
)

check(
    "el controlador termina al encontrar un ganador",
    terminal(final_state)
)

check(
    "P1 gana la partida de prueba",
    winner(final_state) == P1
)


Turno número 1
   0 1 2 3 4 5 6 7 8 9
 0 . . . . . . . . . .
 1 . . . . . . . . . .
 2 . . . . . . . . . .
 3 . . . . . . . . . .
 4 . . . . . . . . . .
 5 . . . . . . . . B .
 6 . . . . . . . . B B
 7 . . . . . . . B B B
 8 . . . . . . B B B B
 9 . . . . . B B B B B
Turno: P1
P1 juega: ((5, 8), (5, 9))

Partida terminada
   0 1 2 3 4 5 6 7 8 9
 0 . . . . . . . . . .
 1 . . . . . . . . . .
 2 . . . . . . . . . .
 3 . . . . . . . . . .
 4 . . . . . . . . . .
 5 . . . . . . . . . B
 6 . . . . . . . . B B
 7 . . . . . . . B B B
 8 . . . . . . B B B B
 9 . . . . . B B B B B
Ganador: P1
Cantidad de turnos: 1
PASS  el controlador termina al encontrar un ganador
PASS  P1 gana la partida de prueba


True

# Parte 4: Interfaz

In [45]:
%pip install --upgrade pygame-ce

Note: you may need to restart the kernel to use updated packages.


In [48]:
import pygame

print(pygame.version.ver)


2.5.8


## Ventana y tablero

In [49]:
import pygame

pygame.init()

CELL_SIZE = 64
BOARD_PIXELS = BOARD_SIZE * CELL_SIZE
PANEL_WIDTH = 260

WINDOW_WIDTH = BOARD_PIXELS + PANEL_WIDTH
WINDOW_HEIGHT = BOARD_PIXELS

BACKGROUND_COLOR = (238, 232, 213)
GRID_COLOR = (70, 70, 70)
P1_CAMP_COLOR = (190, 220, 255)
P2_CAMP_COLOR = (255, 205, 190)
EMPTY_CELL_COLOR = (245, 240, 220)

#### Cuadrícula

In [50]:
def draw_pygame_grid(screen):
    """Dibuja el tablero y resalta ambos campamentos."""
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            position = (row, col)

            if position in P1_CAMP:
                color = P1_CAMP_COLOR
            elif position in P2_CAMP:
                color = P2_CAMP_COLOR
            else:
                color = EMPTY_CELL_COLOR

            rectangle = pygame.Rect(
                col * CELL_SIZE,
                row * CELL_SIZE,
                CELL_SIZE,
                CELL_SIZE
            )

            pygame.draw.rect(
                screen,
                color,
                rectangle
            )

            pygame.draw.rect(
                screen,
                GRID_COLOR,
                rectangle,
                width=1
            )

In [51]:
def preview_pygame_grid():
    """Abre una ventana para comprobar el tablero."""
    screen = pygame.display.set_mode(
        (WINDOW_WIDTH, WINDOW_HEIGHT)
    )

    pygame.display.set_caption(
        "Hoppers - BYD vs. Land Cruiser"
    )

    clock = pygame.time.Clock()
    running = True

    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        screen.fill(BACKGROUND_COLOR)
        draw_pygame_grid(screen)

        pygame.display.flip()
        clock.tick(60)

    pygame.quit()

In [52]:
preview_pygame_grid()